# Phân tích hội tụ của Robust Risk-Aware Bandit
Notebook này nạp dữ liệu từ các file `.npz` và vẽ đồ thị chi tiết cho từng chỉ số.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

# 1. Cấu hình đường dẫn dữ liệu
data_dir = "results/robust_syn_d=20_a=30_pi=eps-greedy0.1_std=0.1"
n_list = [100, 200, 500, 1000, 2000, 5000, 10000]

if not os.path.exists(data_dir):
    potential_dirs = glob.glob("results/robust_syn*")
    if potential_dirs:
        data_dir = potential_dirs[0]
        print(f"Tự động chọn thư mục: {data_dir}")
    else:
        print("Không tìm thấy thư mục results!")

def load_summary_data(data_dir, n_list):
    summary = {'n_regret': [], 'n_extra': [], 'regret_mean': [], 'regret_std': [], 
               'subopt_mean': [], 'subopt_std': [], 'gt_cvar_mean': [], 'gt_cvar_std': [],
               'oracle_cvar': [], 'gt_mean': [], 'gt_var': []}
    
    for n in n_list:
        pattern = os.path.join(data_dir, f"*_n={n}_layers=*.npz")
        files = glob.glob(pattern)
        if not files: continue
            
        d = np.load(files[0], allow_pickle=True)
        algo_name = d['algo_names'][0].replace(' ', '_') if 'algo_names' in d else None

        def get_d(s): 
            if algo_name and f"{algo_name}_{s}" in d: return d[f"{algo_name}_{s}"]
            return d[s] if s in d else None

        # Regret
        r = get_d('regrets')
        if r is not None:
            summary['n_regret'].append(n)
            summary['regret_mean'].append(np.mean(r))
            summary['regret_std'].append(np.std(r) / np.sqrt(len(r.flatten())))
        
        # Stats
        c = get_d('gt_cvars')
        if c is not None:
            summary['n_extra'].append(n)
            o_cvar = np.mean(d['oracle_cvars'])
            summary['gt_cvar_mean'].append(np.mean(c))
            summary['gt_cvar_std'].append(np.std(c) / np.sqrt(len(c.flatten())))
            
            # Subopt
            sub = o_cvar - c
            summary['subopt_mean'].append(np.mean(sub))
            summary['subopt_std'].append(np.std(sub) / np.sqrt(len(c.flatten())))
            summary['oracle_cvar'].append(o_cvar)
            
    return summary

summary = load_summary_data(data_dir, n_list)
print("Dữ liệu đã nạp xong!")

### 1. Biểu đồ Regret (Hội tụ kỳ vọng)

In [ ]:
plt.figure(figsize=(10, 6))
plt.errorbar(summary['n_regret'], summary['regret_mean'], yerr=summary['regret_std'], 
             fmt='-o', capsize=5, label='Agent Regret', markersize=8)
plt.xscale('log'); plt.yscale('log')
plt.title("Expected Regret vs Sample Size (Log-Log)")
plt.xlabel("n"); plt.ylabel("Regret")
plt.grid(True, which="both", alpha=0.3)
plt.legend(); plt.show()

### 2. Suboptimality Gap (Hội tụ rủi ro CVaR)

In [ ]:
plt.figure(figsize=(10, 6))
plt.errorbar(summary['n_extra'], summary['subopt_mean'], yerr=summary['subopt_std'], 
             fmt='-s', color='orange', capsize=5, label='Suboptimality Gap', markersize=8)
plt.axhline(0, color='black', linestyle='--')
plt.title("Suboptimality Gap (Oracle CVaR - Agent CVaR)")
plt.xlabel("n"); plt.ylabel("Gap")
plt.grid(True, alpha=0.3)
plt.legend(); plt.show()

### 3. Agent CVaR vs Oracle CVaR

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(summary['n_extra'], summary['gt_cvar_mean'], 'd-', label='Agent CVaR', color='red', markersize=8)
plt.plot(summary['n_extra'], summary['oracle_cvar'], '--', label='Oracle CVaR', color='black')
plt.fill_between(summary['n_extra'], 
                 np.array(summary['gt_cvar_mean']) - np.array(summary['gt_cvar_std']), 
                 np.array(summary['gt_cvar_mean']) + np.array(summary['gt_cvar_std']), alpha=0.15, color='red')
plt.title("Convergence of Risk-Aware Performance")
plt.xlabel("n"); plt.ylabel("CVaR")
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()